# Monte Carlo Tree Search (MCTS) with UCT
This notebook introduces the core ideas behind Monte Carlo Tree Search (MCTS)
using a minimal working example based on UCT (Upper Confidence Bound applied to Trees).

You will implement:
- UCT-based **Selection**
- Node **Expansion**
- **Rollout** simulation
- **Backpropagation** of returns
- Tracking visit counts and Q-values
- Updating action preferences at the root node

We will follow the four classical MCTS phases.


In [58]:
import math
import random

# Good research should be reproducible - lets get a random seed
random.seed(42)


In [59]:
# Consider the following initial problem situation
Q = {("s0", "a1"): 0.55,
     ("s0", "a2"): 0.40}

N = {"s0": 20}   #Total number of visits for the state
Nsa = {("s0", "a1"): 12,
       ("s0", "a2"): 8}  # visit counts per action in the state

c = math.sqrt(2)   # some exploration constant


In [60]:
def uct(s, a):
    """Will help compute the UCT score for action a in state s."""
    return Q[(s, a)] + c * math.sqrt(math.log(N[s]) / Nsa[(s, a)])


### Task 1:
write code to print the UCT score for both a1 and a2 at s0. Further, print which action, UCT  selects


In [61]:
a1_score = uct("s0", "a1")
a2_score = uct("s0", "a2")

print(f"UCT(s0, a1) = {a1_score:.4f}")
print(f"UCT(s0, a2) = {a2_score:.4f}")

# Pick actiuon with the higher score
if a1_score >= a2_score:
    print("UCT selects a1")
else:
    print("UCT selects a2")

UCT(s0, a1) = 1.2566
UCT(s0, a2) = 1.2654
UCT selects a2


As discussed in class, each iteration of MCTS consists of four key phases:

 * Selection: Starting at the root, select child nodes  until a leaf is reached using a balance between exploration and exploitation.
 * Expansion: Add one or more new child nodes (previously unvisited states).
 * Rollout Perform a random or heuristic-guided simulation from the new node to a terminal state to estimate its value.
* Backpropagation Update value estimates and visit counts along the path back to the root.






In [62]:
def run_iteration(iteration_num=None):
    #Run one iteration of MCTS with detailed logging.

    if iteration_num is not None:
        print(f"\n{'=' * 60}")
        print(f"ITERATION {iteration_num}")
        print(f"{'=' * 60}")

    # (1) --- Selection ---
    if iteration_num is not None:
        print("\n(1) SELECTION:")
        print(f"  Current Q-values: Q(s0,a1)={Q[('s0','a1')]:.4f}, Q(s0,a2)={Q[('s0','a2')]:.4f}")
        print(f"  Current visit counts: N(s0,a1)={Nsa[('s0','a1')]}, N(s0,a2)={Nsa[('s0','a2')]}")
        print(f"  UCT(s0,a1)={uct('s0','a1'):.4f}, UCT(s0,a2)={uct('s0','a2'):.4f}")

    a = max(["a1", "a2"], key=lambda x: uct("s0", x))

    if iteration_num is not None:
        print(f"  Selected action: {a}")

    # (2) --- Expansion ---
    if iteration_num is not None:
        print("\n(2) EXPANSION:")
        print(f"  Expanding from s0 via action {a} to create new child state s_new")

    s_new = "s_new"

    # (3) --- Rollout ---
    R = random.random()  # Return from simulation

    if iteration_num is not None:
        print("\n(3) ROLLOUT:")
        print(f"  Simulating from s_new to terminal state")
        print(f"  Return received: R = {R:.4f}")

    # (4) --- Backpropagation ---
    if iteration_num is not None:
        print("\n(4) BACKPROPAGATION:")
        print(f"  Before update: N(s0)={N['s0']}, N(s0,{a})={Nsa[('s0',a)]}, Q(s0,{a})={Q[('s0',a)]:.4f}")

    N["s0"] += 1
    Nsa[("s0", a)] += 1

    # Incremental mean update:
    Q[(s0 := "s0", a)] += (R - Q[(s0, a)]) / Nsa[(s0, a)]

    if iteration_num is not None:
        print(f"  After update:  N(s0)={N['s0']}, N(s0,{a})={Nsa[('s0',a)]}, Q(s0,{a})={Q[('s0',a)]:.4f}")

    return a, R



**Task 2:**  
Write code to run simulations, before running each iteration document what changes during:
- Selection  
- Expansion  
- Rollout  
- Backpropagation  


In [63]:
for i in range(1, 21):
    run_iteration(i)


ITERATION 1

(1) SELECTION:
  Current Q-values: Q(s0,a1)=0.5500, Q(s0,a2)=0.4000
  Current visit counts: N(s0,a1)=12, N(s0,a2)=8
  UCT(s0,a1)=1.2566, UCT(s0,a2)=1.2654
  Selected action: a2

(2) EXPANSION:
  Expanding from s0 via action a2 to create new child state s_new

(3) ROLLOUT:
  Simulating from s_new to terminal state
  Return received: R = 0.6394

(4) BACKPROPAGATION:
  Before update: N(s0)=20, N(s0,a2)=8, Q(s0,a2)=0.4000
  After update:  N(s0)=21, N(s0,a2)=9, Q(s0,a2)=0.4266

ITERATION 2

(1) SELECTION:
  Current Q-values: Q(s0,a1)=0.5500, Q(s0,a2)=0.4266
  Current visit counts: N(s0,a1)=12, N(s0,a2)=9
  UCT(s0,a1)=1.2623, UCT(s0,a2)=1.2491
  Selected action: a1

(2) EXPANSION:
  Expanding from s0 via action a1 to create new child state s_new

(3) ROLLOUT:
  Simulating from s_new to terminal state
  Return received: R = 0.0250

(4) BACKPROPAGATION:
  Before update: N(s0)=21, N(s0,a1)=12, Q(s0,a1)=0.5500
  After update:  N(s0)=22, N(s0,a1)=13, Q(s0,a1)=0.5096

ITERATION 3

(1

### Task 3:

Write code that will print:
Final Q values,

*   Final Q values
*   Final Visit Counts
*   Preferred at root, s0



In [64]:
print("Final Q values:")
print(f"Q(s0, a1) = {Q[('s0', 'a1')]:.4f}")
print(f"Q(s0, a2) = {Q[('s0', 'a2')]:.4f}")

print("\nFinal visit counts:")
print(f"N(s0) = {N['s0']}")
print(f"N(s0, a1) = {Nsa[('s0', 'a1')]}")
print(f"N(s0, a2) = {Nsa[('s0', 'a2')]}")

# Print preferred at root, s0
if Nsa[("s0", "a1")] >= Nsa[("s0", "a2")]:
    print("\nPreferred action at root s0 is a1")
else:
    print("\nPreferred action at root s0 is a1")

Final Q values:
Q(s0, a1) = 0.4693
Q(s0, a2) = 0.3990

Final visit counts:
N(s0) = 40
N(s0, a1) = 23
N(s0, a2) = 17

Preferred action at root s0 is a1


### Task 4:
Interpret the results:
1. Which action does MCTS end up preferring?
2. Why did the preferred action change or stay the same over the 20 iterations?
3. How does UCT balance exploration and exploitation in your observed results?



Answered in the PDF file

## Optional Challenge:
Extend this notebook by implementing:

1. A real tree structure with:
   - multiple layers,
   - children stored in a dictionary: `children[s] = [list_of_actions]`.

2. A real rollout policy (random or epsilon-greedy).

3. Backpropagation that updates *all* ancestors, not only s0.

4. Visualization of visit counts as a bar chart.
